In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../src")
import pandas as pd
from areas import candidates, build_gazetteer, extract_area
pd.set_option("display.max_colwidth", 90)

In [5]:
# Does the pattern behave on the examples you showed me?
tests = [
    "The attack occurred in Orangi Town.",
    "The incident occurred in the North Nazimabad neighborhood",
    "Incident occurred in the Malir Town area of the city.",
    "The incident occurred in Sector 4-F of Frontier Colony area, near Orangi Town and SITE, Karachi, Sindh.",
    "Near Power House of North Karachi on M A Jinnah Road",
    "Incident occurred on Maripur Road near Lyari",
    "The incident occurred in the Khayaban-e-Rahat of the Defence Housing Authority district in the city.",
    "Occurred in the South district of the city",
]
for t in tests:
    print(candidates(t), "<-", t[:60])

['Orangi'] <- The attack occurred in Orangi Town.
['North Nazimabad'] <- The incident occurred in the North Nazimabad neighborhood
['Malir'] <- Incident occurred in the Malir Town area of the city.
['Frontier Colony', 'Orangi'] <- The incident occurred in Sector 4-F of Frontier Colony area,
['Power House', 'North Karachi', 'M A Jinnah Road'] <- Near Power House of North Karachi on M A Jinnah Road
['Maripur Road', 'Lyari'] <- Incident occurred on Maripur Road near Lyari
['Khayaban-e-Rahat', 'Defence Housing Authority'] <- The incident occurred in the Khayaban-e-Rahat of the Defence
[] <- Occurred in the South district of the city


In [6]:
df = pd.read_parquet("../data/processed/karachi_events_clean.parquet")
gaz = build_gazetteer(df)
print("Gazetteer size:", len(gaz))

res = df.apply(lambda r: extract_area(r["location"], r["summary_clean"], r["city_raw"], gaz),
               axis=1, result_type="expand")
df["area"], df["area_method"] = res[0], res[1]

print(df["area_method"].value_counts(dropna=False))
print("Coverage:", df["area"].notna().mean().round(2))
print(df["area"].value_counts().head(30))

Gazetteer size: 157
area_method
location_pattern     1052
None                  410
summary_gazetteer     176
city_field             26
Name: count, dtype: int64
Coverage: 0.75
area
Orangi                       134
Gulshan-e-Iqbal               66
Korangi                       64
North Nazimabad               63
Lyari                         54
Nazimabad                     47
Baldia                        46
Malir                         39
Landhi                        36
Saddar                        33
Liaquatabad                   30
Qasba Colony                  25
Gulshan                       22
Gulberg                       20
New Karachi                   20
Manghopir                     20
Sohrab Goth                   20
Gulistan-e-Jauhar             19
Quaidabad                     18
Jamshed                       16
Clifton                       14
North Karachi                 14
Abbas                         12
Surjani                       12
Defence Housing Authority 

In [7]:
# Eyeball 20 random results, then the events with no area
print(df.sample(20, random_state=5)[["location", "area", "area_method"]].to_string())
print()
print(df.loc[df["area"].isna(), "summary_clean"].sample(6, random_state=1).str[:160].to_string())
print()
print(df[df["city_raw"].isin(["Hyderabad", "Unknown"])][["eventid", "summary_clean"]].to_string())

                                                                                                             location                          area        area_method
1323                                                          Incident occurred in the Ittehad Town area of the city.                       Ittehad   location_pattern
1091  The incident occurred within the vicinity of Jinnah International Airport at Natha Khan Bridge, Karachi, Sindh.  Jinnah International Airport   location_pattern
1631                                                       The incident occurred in the Orangi Town area of the city.                        Orangi   location_pattern
1107                                                                    The incident occurred in Korangi neighborhood                       Korangi   location_pattern
509                                                 The event occurred in the Sorab Goth area of the city's district.                         Gadap         city_fiel